# Statistical Analysis and Hypothesis Testing

This notebook demonstrates fundamental statistical tests in Python using `scipy.stats`:
1. Normality testing (Shapiro-Wilk)
2. Independent samples t-test
3. One-way ANOVA
4. Chi-squared test of independence
5. Visualizing test results

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_style('whitegrid')
%matplotlib inline
np.random.seed(42)

## Load Data

We use the tips dataset for real-world hypothesis testing scenarios.

In [ ]:
tips = sns.load_dataset('tips')
tips['tip_pct'] = tips['tip'] / tips['total_bill'] * 100
print(f"Shape: {tips.shape}")
tips.head()

## 1. Normality Testing

Many parametric tests assume normally distributed data. The Shapiro-Wilk test checks this assumption.

- H0: Data is normally distributed
- H1: Data is not normally distributed

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ['total_bill', 'tip', 'tip_pct']):
    data = tips[col]
    stat, p = stats.shapiro(data)
    sns.histplot(data, kde=True, ax=ax)
    ax.set_title(f'{col}\nShapiro p={p:.4f} {"(Normal)" if p > 0.05 else "(Not Normal)"}')

plt.tight_layout()
plt.show()

## 2. Independent Samples T-Test

**Question:** Do smokers and non-smokers tip differently (as a percentage)?

- H0: Mean tip percentage is the same for smokers and non-smokers
- H1: Mean tip percentage differs between the two groups

In [ ]:
smoker_tips = tips[tips['smoker'] == 'Yes']['tip_pct']
nonsmoker_tips = tips[tips['smoker'] == 'No']['tip_pct']

t_stat, p_value = stats.ttest_ind(smoker_tips, nonsmoker_tips)

print(f"Smoker mean tip %:     {smoker_tips.mean():.2f} (n={len(smoker_tips)})")
print(f"Non-smoker mean tip %: {nonsmoker_tips.mean():.2f} (n={len(nonsmoker_tips)})")
print(f"\nt-statistic: {t_stat:.4f}")
print(f"p-value:     {p_value:.4f}")
print(f"\nResult: {'Reject H0 - significant difference' if p_value < 0.05 else 'Fail to reject H0 - no significant difference'}")

fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=tips, x='smoker', y='tip_pct', ax=ax)
ax.set_title(f'Tip % by Smoker Status (t={t_stat:.2f}, p={p_value:.4f})')
ax.set_ylabel('Tip Percentage')
plt.tight_layout()
plt.show()

## 3. One-Way ANOVA

**Question:** Does the day of the week affect tip percentage?

- H0: Mean tip percentage is the same across all days
- H1: At least one day has a different mean tip percentage

In [ ]:
groups = [group['tip_pct'].values for name, group in tips.groupby('day')]
f_stat, p_value = stats.f_oneway(*groups)

print("Mean tip % by day:")
print(tips.groupby('day')['tip_pct'].agg(['mean', 'std', 'count']).round(2))
print(f"\nF-statistic: {f_stat:.4f}")
print(f"p-value:     {p_value:.4f}")
print(f"\nResult: {'Reject H0 - significant difference' if p_value < 0.05 else 'Fail to reject H0 - no significant difference'}")

fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=tips, x='day', y='tip_pct', ax=ax, inner='quartile')
ax.set_title(f'Tip % by Day (ANOVA F={f_stat:.2f}, p={p_value:.4f})')
ax.set_ylabel('Tip Percentage')
plt.tight_layout()
plt.show()

## 4. Chi-Squared Test of Independence

**Question:** Is there an association between smoker status and meal time (lunch vs dinner)?

- H0: Smoker status and meal time are independent
- H1: There is an association between smoker status and meal time

In [ ]:
contingency = pd.crosstab(tips['smoker'], tips['time'])
print("Contingency Table:")
print(contingency)

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"\nExpected frequencies:")
print(pd.DataFrame(expected, index=contingency.index, columns=contingency.columns).round(1))
print(f"\nChi-squared: {chi2:.4f}")
print(f"p-value:     {p_value:.4f}")
print(f"Degrees of freedom: {dof}")
print(f"\nResult: {'Reject H0 - variables are associated' if p_value < 0.05 else 'Fail to reject H0 - variables are independent'}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
contingency.plot(kind='bar', ax=axes[0])
axes[0].set_title('Observed Frequencies')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

pd.DataFrame(expected, index=contingency.index, columns=contingency.columns).plot(kind='bar', ax=axes[1])
axes[1].set_title('Expected Frequencies (under H0)')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

## Summary of Results

| Test | Question | p-value | Conclusion |
|------|----------|---------|------------|
| Shapiro-Wilk | Is tip_pct normal? | varies | Check output above |
| T-test | Smoker vs non-smoker tips? | see above | Depends on p < 0.05 |
| ANOVA | Day affect tip %? | see above | Depends on p < 0.05 |
| Chi-squared | Smoker vs meal time? | see above | Depends on p < 0.05 |

**Key takeaway:** Always check assumptions (normality, equal variance) before choosing a test, and consider effect size alongside p-values.